In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import warnings

warnings.filterwarnings('ignore')

def backtest_financeiro_hmm_consolidado():
    print("=== BACKTEST FINANCEIRO: MODELO HMM + EWMA (BASE ÚNICA CONSOLIDADA CSV) ===")
    
    # -------------------------------------------------------------------------
    # 1. Caminhos Dinâmicos
    # -------------------------------------------------------------------------
    diretorio_dados = r"c:\VSCodeWorkspace\TCC_Escrito\data"
    custo_corretagem_spread = 0.005
    
    # -------------------------------------------------------------------------
    # 2. Leitura da Base Única Consolidada em CSV
    # -------------------------------------------------------------------------
    print("1. A carregar a base mestre consolidada em CSV (Ações + IBOV + CDI)...")
    caminho_mestre = os.path.join(diretorio_dados, "base_mestre_consolidada_IBOV_CDI_SELIC.csv")
    
    # Lemos o CSV no padrão brasileiro
    df_mestre = pd.read_csv(caminho_mestre, sep=';', decimal=',', encoding='latin1')
    
    # =========================================================================
    # A VERDADEIRA BLINDAGEM DE DATAS (BASE MESTRE)
    # =========================================================================
    # 1. Pega no nome exato da primeira coluna do seu CSV (que contém as datas)
    coluna_data = df_mestre.columns[0]
    
    # 2. Converte para Data e define como Índice da tabela
    df_mestre[coluna_data] = pd.to_datetime(df_mestre[coluna_data], errors='coerce')
    df_mestre.set_index(coluna_data, inplace=True)
    df_mestre.index.name = 'Data'
    
    # 3. Normaliza (tira horas), remove fusos horários e organiza cronologicamente
    df_mestre.index = df_mestre.index.normalize()
    if df_mestre.index.tz is not None:
        df_mestre.index = df_mestre.index.tz_localize(None)
        
    df_mestre = df_mestre.sort_index()
    # 4. Remove dias duplicados (mantém sempre a última cotação do dia)
    df_mestre = df_mestre[~df_mestre.index.duplicated(keep='last')]
        
    # =========================================================================
    # TRATAMENTO DE DADOS: GARANTIR QUE TUDO É NÚMERO
    # =========================================================================
    print("-> A converter colunas de texto para formato financeiro numérico...")
    for coluna in df_mestre.columns:
        if df_mestre[coluna].dtype == 'object' or df_mestre[coluna].dtype.name in ['string', 'string[pyarrow]']:
            df_mestre[coluna] = df_mestre[coluna].astype(str).str.replace(',', '.')
        
        df_mestre[coluna] = pd.to_numeric(df_mestre[coluna], errors='coerce')
    
    # -------------------------------------------------------------------------
    # 3. Separação de Ações e Variáveis Macro
    # -------------------------------------------------------------------------
    colunas_macro = ['IBOV', 'CDI', 'SELIC']
    colunas_ativos = [col for col in df_mestre.columns if col not in colunas_macro]
    
    print("2. A calcular os retornos diários e alinhar as taxas...")
    df_retornos_ativos = df_mestre[colunas_ativos].pct_change().dropna(how='all')
    retornos_ibov = df_mestre['IBOV'].pct_change().dropna(how='all')
    
    # O CDI já é taxa diária, não precisa de pct_change()
    retornos_cdi = df_mestre['CDI'].loc[df_retornos_ativos.index].fillna(0)
    
    # -------------------------------------------------------------------------
    # 4. Leitura do Histórico de Pesos
    # -------------------------------------------------------------------------
    print("3. A carregar o histórico de pesos da estratégia...")
    caminho_pesos_csv = os.path.join(diretorio_dados, "historico_alocacao_HMM_EWMA.csv")
    caminho_pesos_parquet = os.path.join(diretorio_dados, "historico_alocacao_HMM_EWMA.parquet")
    
    # Tenta ler o arquivo de pesos que existir
    if os.path.exists(caminho_pesos_csv):
        # Tenta ler com ; primeiro, se ficar só 1 coluna, troca para ,
        df_pesos = pd.read_csv(caminho_pesos_csv, sep=';', decimal=',')
        if len(df_pesos.columns) < 5:
            df_pesos = pd.read_csv(caminho_pesos_csv, sep=',', decimal='.')
    else:
        df_pesos = pd.read_parquet(caminho_pesos_parquet)
    
    # =========================================================================
    # BLINDAGEM DE DATAS (ARQUIVO DE PESOS)
    # =========================================================================
    if not isinstance(df_pesos.index, pd.DatetimeIndex):
        coluna_data_pesos = df_pesos.columns[0]
        df_pesos[coluna_data_pesos] = pd.to_datetime(df_pesos[coluna_data_pesos], errors='coerce')
        df_pesos.set_index(coluna_data_pesos, inplace=True)
        
    df_pesos.index.name = 'Data'
    df_pesos.index = df_pesos.index.normalize()
    
    if df_pesos.index.tz is not None:
        df_pesos.index = df_pesos.index.tz_localize(None)
        
    df_pesos = df_pesos.sort_index()
    df_pesos = df_pesos[~df_pesos.index.duplicated(keep='last')]
    
    # Força os pesos a serem números
    for coluna in df_pesos.columns:
        if df_pesos[coluna].dtype == 'object' or df_pesos[coluna].dtype.name in ['string', 'string[pyarrow]']:
            df_pesos[coluna] = df_pesos[coluna].astype(str).str.replace(',', '.')
        df_pesos[coluna] = pd.to_numeric(df_pesos[coluna], errors='coerce')
    
    # -------------------------------------------------------------------------
    # 5. Cruzamento Temporal (Shift Sem Look-ahead Bias)
    # -------------------------------------------------------------------------
    print("4. A cruzar a alocação do HMM com os retornos reais do mercado...")
    data_inicio_backtest = df_pesos.index[0]
    df_retornos_ativos = df_retornos_ativos.loc[data_inicio_backtest:]
    retornos_ibov = retornos_ibov.loc[data_inicio_backtest:]
    retornos_cdi = retornos_cdi.loc[data_inicio_backtest:]
    
    # Desloca a alocação em 1 dia para o retorno ser calculado apenas no dia seguinte
    pesos_diarios = df_pesos.reindex(df_retornos_ativos.index).ffill().shift(1)
    pesos_diarios = pesos_diarios.dropna(how='all')
    
    print(f"   -> Cruzamento concluído! O backtest terá {len(pesos_diarios)} dias úteis.")
    
    df_retornos_ativos = df_retornos_ativos.loc[pesos_diarios.index]
    retornos_ibov = retornos_ibov.loc[pesos_diarios.index]
    retornos_cdi = retornos_cdi.loc[pesos_diarios.index]
    
    # -------------------------------------------------------------------------
    # 6. Cálculo Financeiro e Custos
    # -------------------------------------------------------------------------
    retorno_bruto_portfolio = (pesos_diarios * df_retornos_ativos).sum(axis=1)
    giro_diario_carteira = pesos_diarios.diff().abs().sum(axis=1).fillna(0)
    custo_transacao_diario = giro_diario_carteira * custo_corretagem_spread
    
    retorno_liquido_portfolio = retorno_bruto_portfolio - custo_transacao_diario
    
    # -------------------------------------------------------------------------
    # 7. Composição e Equity Curve
    # -------------------------------------------------------------------------
    patrimonio_hmm = (1 + retorno_liquido_portfolio).cumprod() - 1
    patrimonio_ibov = (1 + retornos_ibov).cumprod() - 1
    patrimonio_cdi = (1 + retornos_cdi).cumprod() - 1
    
    df_patrimonio = pd.DataFrame({
        'Carteira_HMM_EWMA': patrimonio_hmm,
        'IBOV_Mercado': patrimonio_ibov,
        'CDI_RendaFixa': patrimonio_cdi
    })

    # -------------------------------------------------------------------------
    # 8. Renderização do Gráfico Final
    # -------------------------------------------------------------------------
    print("5. A renderizar o Gráfico da Vitória...")
    plt.figure(figsize=(14, 7))
    plt.plot(df_patrimonio.index, df_patrimonio['Carteira_HMM_EWMA'], label='Modelo Avançado (HMM + EWMA)', color='purple', linewidth=2)
    plt.plot(df_patrimonio.index, df_patrimonio['IBOV_Mercado'], label='IBOVESPA', color='red', linestyle='--', linewidth=1.5)
    plt.plot(df_patrimonio.index, df_patrimonio['CDI_RendaFixa'], label='CDI', color='green', linestyle=':', linewidth=2)
    
    plt.title('Performance Final: Modelo de Regimes de Markov vs Mercado (2015-2025)', fontsize=16)
    plt.xlabel('Ano', fontsize=12)
    plt.ylabel('Retorno Acumulado (%)', fontsize=12)
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.legend(loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3)
    
    caminho_grafico = os.path.join(diretorio_dados, "equity_curve_hmm_final.png")
    plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
    plt.close()
    
    # KPIs Finais
    ret_total_hmm = df_patrimonio['Carteira_HMM_EWMA'].iloc[-1]
    ret_total_ibov = df_patrimonio['IBOV_Mercado'].iloc[-1]
    ret_total_cdi = df_patrimonio['CDI_RendaFixa'].iloc[-1]
    
    print("\n🏆 === RESULTADO FINAL DO TCC (2015 a 2025) === 🏆")
    print(f"Retorno Acumulado HMM+EWMA: {ret_total_hmm:.2%}")
    print(f"Retorno Acumulado IBOVESPA: {ret_total_ibov:.2%}")
    print(f"Retorno Acumulado CDI:      {ret_total_cdi:.2%}")
    print(f"\nGráfico da Vitória exportado para: {caminho_grafico}")
    print("==================================================")

    return df_patrimonio

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
if __name__ == "__main__":
    resultado_final = backtest_financeiro_hmm_consolidado()

=== BACKTEST FINANCEIRO: MODELO HMM + EWMA (BASE ÚNICA CONSOLIDADA CSV) ===
1. A carregar a base mestre consolidada em CSV (Ações + IBOV + CDI)...
-> A converter colunas de texto para formato financeiro numérico...
2. A calcular os retornos diários e alinhar as taxas...
3. A carregar o histórico de pesos da estratégia...
4. A cruzar a alocação do HMM com os retornos reais do mercado...
   -> Cruzamento concluído! O backtest terá 2707 dias úteis.
5. A renderizar o Gráfico da Vitória...

🏆 === RESULTADO FINAL DO TCC (2015 a 2025) === 🏆
Retorno Acumulado HMM+EWMA: 381.44%
Retorno Acumulado IBOVESPA: 243.49%
Retorno Acumulado CDI:      171.10%

Gráfico da Vitória exportado para: c:\VSCodeWorkspace\TCC_Escrito\data\equity_curve_hmm_final.png
